In [1]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

/NFSHOME/adangelo/miniconda3/envs/representer/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [120]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

2025-05-05 23:08:30,-1572379330 | INFO | 2638062 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc
2025-05-05 23:08:30,-1572379326 | INFO | 2638062 - Setting seeds to: 0
2025-05-05 23:08:30,-1572379323 | INFO | 2638062 - REMOVAL TYPE SET AS edge
2025-05-05 23:08:30,-1572379322 | INFO | 2638062 - Caching System: False.
2025-05-05 23:08:30,-1572379320 | INFO | 2638062 - Instantiating: torch_geometric.datasets.Planetoid
2025-05-05 23:08:30,-1572379294 | INFO | 2638062 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
2025-05-05 23:08:30,-1572379293 | INFO | 2638062 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'PubMed'}, 'preprocess': []}}}
2025-05-05 23:08:30,-1572379263 | INFO | 2638062 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSpl

2025-05-05 23:08:30,-1572379213 | INFO | 2638062 - ['all', 'all_shuffled', '-']
2025-05-05 23:08:30,-1572379212 | INFO | 2638062 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
2025-05-05 23:08:30,-1572379211 | INFO | 2638062 - ['all', 'all_shuffled', '-', 'train', 'test']
2025-05-05 23:08:30,-1572379210 | INFO | 2638062 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
2025-05-05 23:08:30,-1572379180 | INFO | 2638062 - ['all', 'all_shuffled', '-', 'train', 'test', 'forget', 'retain']
2025-05-05 23:08:30,-1572379179 | INFO | 2638062 - ['all', 'all_shuffled', '-', 'train', 'test', 'forget', 'retain']
2025-05-05 23:08:30,-1572379178 | INFO | 2638062 - Created Configurable: erasure.data.datasets.DatasetManager.DatasetManager


In [121]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

2025-05-05 23:08:30,-1572379100 | INFO | 2638062 - Instantiating: erasure.model.graphs.GCN.GCN


2025-05-05 23:08:30,-1572379094 | INFO | 2638062 - Instantiating: torch.optim.Adam
2025-05-05 23:08:30,-1572379093 | INFO | 2638062 - Instantiating: torch.nn.CrossEntropyLoss
graph has edges  torch.Size([2, 88648])
2025-05-05 23:08:30,-1572379007 | INFO | 2638062 - epoch = 0 ---> loss = 1.0973	 accuracy = 0.4030
graph has edges  torch.Size([2, 88648])
2025-05-05 23:08:30,-1572378932 | INFO | 2638062 - epoch = 1 ---> loss = 1.0946	 accuracy = 0.4357
graph has edges  torch.Size([2, 88648])
2025-05-05 23:08:30,-1572378868 | INFO | 2638062 - epoch = 2 ---> loss = 1.0921	 accuracy = 0.4503
graph has edges  torch.Size([2, 88648])
2025-05-05 23:08:30,-1572378806 | INFO | 2638062 - epoch = 3 ---> loss = 1.0894	 accuracy = 0.4661
graph has edges  torch.Size([2, 88648])
2025-05-05 23:08:30,-1572378746 | INFO | 2638062 - epoch = 4 ---> loss = 1.0867	 accuracy = 0.4947
graph has edges  torch.Size([2, 88648])
2025-05-05 23:08:30,-1572378685 | INFO | 2638062 - epoch = 5 ---> loss = 1.0839	 accuracy 

In [122]:
data_manager.partitions['all'][0][0]

Data(x=[19717, 500], edge_index=[2, 88648])

In [123]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

graph tensor([], size=(2, 0), dtype=torch.int64)
2025-05-05 23:08:37,-1572371602 | INFO | 2638062 - Created Configurable: erasure.unlearners.GoldModel.GoldModelGraph
2025-05-05 23:08:38,-1572371410 | INFO | 2638062 - Created Configurable: erasure.unlearners.composite.Identity


In [124]:
data_manager.partitions['all'][0][0]

Data(x=[19717, 500], edge_index=[2, 88648])

In [125]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[19717, 500], edge_index=[2, 0])

In [126]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

2025-05-05 23:08:38,-1572370543 | INFO | 2638062 - Created Configurable: erasure.evaluations.running.RunTime
2025-05-05 23:08:38,-1572370541 | INFO | 2638062 - Function: <function accuracy_score at 0x7fbb4bf044c0>
2025-05-05 23:08:38,-1572370541 | INFO | 2638062 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph


2025-05-05 23:08:38,-1572370521 | INFO | 2638062 - Function: <function accuracy_score at 0x7fbb4bf044c0>
2025-05-05 23:08:38,-1572370511 | INFO | 2638062 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
2025-05-05 23:08:38,-1572370510 | INFO | 2638062 - Created Configurable: erasure.evaluations.measures.SaveValues
2025-05-05 23:08:38,-1572370509 | INFO | 2638062 - Created Configurable: erasure.evaluations.manager.Evaluator
2025-05-05 23:08:38,-1572370490 | INFO | 2638062 - ####		 Evaluating Unlearner GoldModelGraph 		####
2025-05-05 23:08:39,-1572370295 | INFO | 2638062 - Unlearning copyed predictor: <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7fbb30a58cd0>
2025-05-05 23:08:39,-1572370294 | INFO | 2638062 - Instantiating: erasure.model.graphs.GCN.GCN
2025-05-05 23:08:39,-1572370292 | INFO | 2638062 - Instantiating: torch.optim.Adam
2025-05-05 23:08:39,-1572370291 | INFO | 2638062 - Instantiating: torch.nn.CrossEntropyLoss
graph has edges  torch.Size